In [ ]:
# installs
!pip install -U trl
!pip install -U peft
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
# imports
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

# get huggingface token
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

In [ ]:
# get dataset and preprocessing
train_dataset = load_dataset("SetFit/sst2", split = "train", token = hf_token)
test_dataset = load_dataset("SetFit/sst2", split = "validation", token = hf_token)
pred_dataset = load_dataset("SetFit/sst2", split = "test", token = hf_token)

Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.


In [ ]:
# initialize model
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased",
                                                           token = hf_token,
                                                           device_map = "auto",)
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased",
                                          token = hf_token)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# finetuning step
peft_config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    lora_dropout = 0.05,
    target_modules = ["q_lin", "v_lin"],
    task_type = "SEQ_CLS",
)

train_args = TrainingArguments(
    num_train_epochs = 1
)

def tokenize_fn(example):
  return tokenizer(example["text"], truncation = True)
train_dataset_with_tokens = train_dataset.map(tokenize_fn, batched = True)
test_dataset_with_tokens = test_dataset.map(tokenize_fn, batched = True)

model = get_peft_model(model, peft_config)
trainer = Trainer(
    model=model,
    processing_class = tokenizer,
    train_dataset = train_dataset_with_tokens,
    eval_dataset = test_dataset_with_tokens,
    args = train_args,
)

trainer.train()

Map:   0%|          | 0/6920 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Step,Training Loss
500,0.454790


TrainOutput(global_step=865, training_loss=0.41863084032356396, metrics={'train_runtime': 23.7085, 'train_samples_per_second': 291.878, 'train_steps_per_second': 36.485, 'total_flos': 78216225675264.0, 'train_loss': 0.41863084032356396, 'epoch': 1.0})

In [ ]:
# evaluate model
eval_results = trainer.evaluate()
print(eval_results)

Training Loss,Validation Loss,Step
0.454790,0.345756,865


{'eval_loss': 0.3457559049129486}


In [ ]:
# predict on a dataset
import numpy as np
import pandas as pd

pred_dataset_with_tokens = pred_dataset.map(tokenize_fn, batched = True)
preds = trainer.predict(pred_dataset_with_tokens) # outputs logits, not final label.
predicted_classes = np.argmax(preds.predictions, axis=-1)
df = pd.DataFrame({
    "predicted_label": predicted_classes
})
if preds.label_ids is not None:
  df["true_label"] = preds.label_ids
df["text"] = pred_dataset["text"]

df.to_csv("classification_results.csv", index=False)